# Imports

In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../../")

In [2]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../../")
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import scipy
import pandas as pd
import cupy as cp
import seaborn as sns
from __future__ import annotations

# from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict

from dataclasses import dataclass

import pyaldata as pyal

import tools.dsp as dsp
from tools.params import Params
import tools.dimensionality as dim
import tools.decoding as decode
import tools.subspaces as subspaces
import tools.viz.utilityTools as vizutils
import tools.dataTools as dt
from tools.params import colors
import tools.reports as reports
import tools.kinematics as kin


from tqdm import tqdm
import pickle

from joblib import Parallel, delayed
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import cross_validate, KFold, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.naive_bayes import GaussianNB



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
sessions = [
    # 'M061_2025_03_04_10_00', 
    # 'M061_2025_03_05_14_00', 
    'M061_2025_03_06_14_00',
    # 'M063_2025_03_13_14_00',
    # 'M063_2025_03_14_15_30', 
    'M062_2025_03_20_14_00',d
    # 'M062_2025_03_21_14_00',
    'M078_2025_08_06_15_00',
    # 'M086_2025_12_10_15_00',
    # 'M103_2026_02_18_15_30',
    # 'M106_2026_02_25_15_00'

]
session_area_exclusions = {
    # 'M062_2025_03_20_14_00': ['VAL'],
    # 'M062_2025_03_21_14_00': ['SSp', 'CP'],
}

In [4]:
all_session_processed = dsp.load_sessions_for_trial_analyses(sessions)

Loading sessions:   0%|          | 0/3 [00:00<?, ?it/s]


M061_2025_03_06_14_00
fields: ['values_before_camera_trigger', 'idx_before_camera_trigger'] could not be converted to int.


/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_X might be a time-varying field. It matches the length of all_spikes on 99.85052316890882% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_X might be a time-varying field. It matches the length of all_spikes on 99.85052316890882% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_Y might be a time-varying field. It matches the length of all_spikes on 99.85052316890882% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_Y might be a time-varying field. It matches the length of all_spikes on 99.85052316890882% of trials
  warnings.warn(


['all_spikes', 'SSp_spikes', 'VAL_spikes', 'GPe_spikes', 'CP_spikes', 'MOp_spikes']
Resulting all_spikes ephys data shape is (NxT): (11, 60000)
Resulting SSp_spikes ephys data shape is (NxT): (69, 60000)
Resulting VAL_spikes ephys data shape is (NxT): (168, 60000)
Resulting GPe_spikes ephys data shape is (NxT): (120, 60000)
Resulting CP_spikes ephys data shape is (NxT): (172, 60000)
Resulting MOp_spikes ephys data shape is (NxT): (178, 60000)
Skipped 40 trials
Otsu immobility threshold: 2.9536
Dropped 48 of 333 rows (14.41%).
  threshold = -0.031  |  trials by rank: 220  |  trials kept: 217


Loading sessions:  33%|███▎      | 1/3 [00:28<00:57, 28.60s/it]

...


M078_2025_08_06_15_00
fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'idx_motion', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
['SSp_spikes', 'all_spikes', 'VAL_spikes', 'MOs_spikes', 'CP_spikes', 'MOp_spikes']


/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_X might be a time-varying field. It matches the length of SSp_spikes on 99.9003984063745% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_X might be a time-varying field. It matches the length of SSp_spikes on 99.9003984063745% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_Y might be a time-varying field. It matches the length of SSp_spikes on 99.9003984063745% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_Y might be a time-varying field. It matches the length of SSp_spikes on 99.9003984063745% of trials
  warnings.warn(


Resulting SSp_spikes ephys data shape is (NxT): (90, 48000)
Resulting all_spikes ephys data shape is (NxT): (54, 48000)
Resulting VAL_spikes ephys data shape is (NxT): (192, 48000)
Resulting MOs_spikes ephys data shape is (NxT): (172, 48000)
Resulting CP_spikes ephys data shape is (NxT): (151, 48000)
Resulting MOp_spikes ephys data shape is (NxT): (83, 48000)
Skipped 107 trials
Otsu immobility threshold: 2.2837
Dropped 83 of 501 rows (16.57%).
  threshold = -0.033  |  trials by rank: 286  |  trials kept: 277


Loading sessions: 100%|██████████| 3/3 [01:56<00:00, 38.72s/it]


Loaded 3/3 sessions successfully.
